In [25]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd


BASE_DIR = '/Users/uddashyakumar/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/y3/Sem 2/COMP3522/PM3'
DATA_DIR = f'{BASE_DIR}/Data'

LATITUDE_KEYS = [
    "latitude",
    "lat",
    "y",
    "ycoord",
    "ycoordinate",
    "northing",
]

LONGITUDE_KEYS = [
    "longitude",
    "long",
    "lon",
    "lng",
    "x",
    "xcoord",
    "xcoordinate",
    "longtitude",
    "easting",
]


def normalize_column_name(name: str) -> str:
    """Normalize column name for robust comparisons."""
    return re.sub(r"[^a-z0-9]", "", name.lower())


def _match_column(columns, candidates):
    normalized = {col: normalize_column_name(col) for col in columns}

    # Prefer exact matches
    for candidate in candidates:
        for col, norm in normalized.items():
            if norm == candidate:
                return col

    # Prefer suffix matches (e.g., mylatitude)
    for candidate in candidates:
        for col, norm in normalized.items():
            if norm.endswith(candidate):
                return col

    # Fallback to substring search
    for candidate in candidates:
        for col, norm in normalized.items():
            if candidate in norm:
                return col

    return None


def detect_lat_lon_columns(df: pd.DataFrame):
    """Return detected latitude and longitude column names or (None, None)."""
    lat_col = _match_column(df.columns, LATITUDE_KEYS)
    lon_col = _match_column(df.columns, LONGITUDE_KEYS)

    if lat_col is None or lon_col is None:
        return None, None
    return lat_col, lon_col


def convert_to_geodf(df: pd.DataFrame, lat_col: str, lon_col: str, source_crs="EPSG:4326", target_crs="EPSG:3414"):
    """Convert DataFrame with coordinates to projected GeoDataFrame."""
    df = df.copy()
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df = df.dropna(subset=[lat_col, lon_col])

    if df.empty:
        # Return empty GeoDataFrame with the target CRS
        return gpd.GeoDataFrame(df, geometry=[], crs=target_crs)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs=source_crs,
    )
    return gdf.to_crs(target_crs)


def load_amenity_geodf(path: Path):
    """Load CSV amenity file and convert to projected GeoDataFrame."""
    df = pd.read_csv(path)
    lat_col, lon_col = detect_lat_lon_columns(df)

    if lat_col is None or lon_col is None:
        print(f"[WARN] {path.name}: latitude/longitude columns not detected. Skipping.")
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:3414")

    gdf = convert_to_geodf(df, lat_col, lon_col)
    if gdf.empty:
        print(f"[WARN] {path.name}: no valid coordinates after cleaning. Skipping.")
    return gdf


def compute_nearest_distance(resale_gdf: gpd.GeoDataFrame, amenity_gdf: gpd.GeoDataFrame, distance_col: str):
    """Compute nearest distance from each resale point to amenity GeoDataFrame."""
    if amenity_gdf is None or amenity_gdf.empty:
        resale_gdf[distance_col] = np.nan
        return

    joined = gpd.sjoin_nearest(
        resale_gdf[["geometry"]],
        amenity_gdf[["geometry"]],
        how="left",
        distance_col=distance_col,
        lsuffix="resale",
        rsuffix="amenity",
    )

    distances = joined.groupby(level=0)[distance_col].min()
    resale_gdf[distance_col] = distances.reindex(resale_gdf.index).to_numpy()


def main():
    amenity_files = [
        ("cycling_paths_latlon.csv", "dist_m_cycling_path"),
        ("disability_services_latlon.csv", "dist_m_disability"),
        ("gyms_latlon.csv", "dist_m_gym"),
        ("hawker_centres_latlon.csv", "dist_m_hawker"),
        ("mrt_station_exits_latlon.csv", "dist_m_mrt_exit"),
        ("parks_latlon.csv", "dist_m_park"),
        ("schools_latlon.csv", "dist_m_school"),
        ("supermarkets_latlon.csv", "dist_m_supermarket"),
        ("taxi_stops_latlon.csv", "dist_m_taxi_stop"),
        ("tourist_attractions_latlon.csv", "dist_m_tourist"),
    ]

    amenities = {}
    for filename, distance_col in amenity_files:
        path = f'{DATA_DIR}/{filename}'
        gdf = load_amenity_geodf(path)
        amenities[distance_col] = gdf

    resale_path = f'{DATA_DIR}/resale_2021plus_geocoded_friend_style.parquet'
    resale_df = pd.read_parquet(resale_path)

    resale_lat_col, resale_lon_col = detect_lat_lon_columns(resale_df)
    if resale_lat_col is None or resale_lon_col is None:
        raise ValueError("Latitude/longitude columns not detected in resale dataset.")

    resale_df["Latitude"] = pd.to_numeric(resale_df[resale_lat_col], errors="coerce")
    resale_df["Longitude"] = pd.to_numeric(resale_df[resale_lon_col], errors="coerce")
    resale_df = resale_df.dropna(subset=["Latitude", "Longitude"])

    resale_df["month"] = pd.to_datetime(resale_df["month"], errors="coerce")
    resale_df = resale_df[resale_df["month"] >= pd.Timestamp("2023-01-01")].copy()
    resale_df["month"] = resale_df["month"].dt.to_period("M").dt.to_timestamp()

    resale_gdf = convert_to_geodf(resale_df, "Latitude", "Longitude")

    for distance_col, amenity_gdf in amenities.items():
        compute_nearest_distance(resale_gdf, amenity_gdf, distance_col)

    keep_columns = [
        "month",
        "town",
        "street_name",
        "floor_area_sqm",
        "resale_price",
        "Latitude",
        "Longitude",
    ] + list(amenities.keys())

    df_final = resale_gdf.drop(columns="geometry").copy()
    missing_cols = [col for col in keep_columns if col not in df_final.columns]
    if missing_cols:
        raise KeyError(f"Missing expected columns in final DataFrame: {missing_cols}")

    df_final = df_final[keep_columns]

    print(f"Rows: {len(df_final)}")
    print("Columns:", df_final.columns.tolist())
    print(df_final.head())

    return df_final


if __name__ == "__main__":
    df_final = main()
    output_path = f"{DATA_DIR}/resale_2021plus_with_amenity_distances.parquet"
    df_final.to_parquet(output_path)
    print(f"Saved final DataFrame to {output_path}")

Rows: 147600
Columns: ['month', 'town', 'street_name', 'floor_area_sqm', 'resale_price', 'Latitude', 'Longitude', 'dist_m_cycling_path', 'dist_m_disability', 'dist_m_gym', 'dist_m_hawker', 'dist_m_mrt_exit', 'dist_m_park', 'dist_m_school', 'dist_m_supermarket', 'dist_m_taxi_stop', 'dist_m_tourist']
            month        town        street_name  floor_area_sqm  \
286616 2023-01-01  ANG MO KIO  ANG MO KIO AVE 10            44.0   
286617 2023-01-01  ANG MO KIO  ANG MO KIO AVE 10            44.0   
286618 2023-01-01  ANG MO KIO   ANG MO KIO AVE 3            44.0   
286619 2023-01-01  ANG MO KIO   ANG MO KIO AVE 3            44.0   
286620 2023-01-01  ANG MO KIO   ANG MO KIO AVE 3            44.0   

        resale_price  Latitude   Longitude  dist_m_cycling_path  \
286616      267000.0  1.366770  103.856590           154.746214   
286617      267000.0  1.366770  103.856590           154.746214   
286618      280000.0  1.369306  103.856036            91.630461   
286619      280000.0  1

In [ ]:
import argparse
import math
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


try:
    from xgboost import XGBRegressor  # type: ignore

    XGB_AVAILABLE = True
except Exception:  # pragma: no cover - optional dependency
    XGB_AVAILABLE = False

try:
    from catboost import CatBoostRegressor  # type: ignore

    CAT_AVAILABLE = True
except Exception:  # pragma: no cover - optional dependency
    CAT_AVAILABLE = False


RANDOM_STATE = 42
# robust BASE_DIR resolution for scripts and notebooks
try:
    BASE_DIR = Path(__file__).resolve().parent
except Exception:
    # running in a notebook: prefer existing BASE_DIR (string) if present, else cwd
    existing = globals().get("BASE_DIR")
    if existing is not None:
        BASE_DIR = Path('/Users/uddashyakumar/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/y3/Sem 2/COMP3522/PM3')
    else:
        BASE_DIR = Path.cwd()
DATA_PATH = Path('/Users/uddashyakumar/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/y3/Sem 2/COMP3522/PM3/Data/resale_2021plus_with_amenity_distances.parquet')
DEFAULT_ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACT_FILES = {
    "results": "results_df.joblib",
    "best_models": "best_models.joblib",
    "lin_coef": "lin_coef_df.joblib",
    "feat_importances": "feat_importances.joblib",
    "splits": "splits.joblib",
}


def parse_args(arg_list: Optional[List[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train or load resale price models.")
    parser.add_argument(
        "--refresh",
        action="store_true",
        help="Force retraining even if cached artifacts are available.",
    )
    parser.add_argument(
        "--artifacts-dir",
        type=str,
        default=str(DEFAULT_ARTIFACTS_DIR),
        help="Directory for cached artifacts (default: %(default)s).",
    )
    return parser.parse_args(arg_list)


def get_artifact_paths(artifacts_dir: Path) -> Dict[str, Path]:
    return {key: artifacts_dir / filename for key, filename in ARTIFACT_FILES.items()}


def save_artifacts(
    paths: Dict[str, Path],
    results_df: pd.DataFrame,
    best_models: Dict[str, dict],
    lin_coef_df: Optional[pd.DataFrame],
    feat_importances: Dict[str, pd.Series],
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
) -> None:
    joblib.dump(results_df, paths["results"])
    joblib.dump(best_models, paths["best_models"])
    joblib.dump(lin_coef_df, paths["lin_coef"])
    joblib.dump(feat_importances, paths["feat_importances"])
    joblib.dump(
        {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test},
        paths["splits"],
    )


def load_artifacts(paths: Dict[str, Path]) -> Optional[Dict[str, object]]:
    if not all(path.exists() for path in paths.values()):
        return None
    return {
        "results_df": joblib.load(paths["results"]),
        "best_models": joblib.load(paths["best_models"]),
        "lin_coef_df": joblib.load(paths["lin_coef"]),
        "feat_importances": joblib.load(paths["feat_importances"]),
        "splits": joblib.load(paths["splits"]),
    }


def make_one_hot_encoder() -> OneHotEncoder:
    """Return a dense one-hot encoder compatible across sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # for older sklearn
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def load_and_prepare_data(path: Path) -> pd.DataFrame:
    """Load parquet data, enforce ordering, and derive temporal features."""
    df = pd.read_parquet(path)

    required_cols = {"resale_price", "Latitude", "Longitude", "month"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise KeyError(f"Missing required columns: {sorted(missing_cols)}")

    df = df.copy()
    df = df.dropna(subset=["resale_price", "Latitude", "Longitude"])

    df["month_dt"] = pd.to_datetime(df["month"], errors="coerce")
    df = df.dropna(subset=["month_dt"])
    df = df.sort_values(["month_dt"]).reset_index(drop=True)

    df["t_idx"] = df["month_dt"].rank(method="dense").astype(int)
    df["qtr"] = df["month_dt"].dt.to_period("Q")

    return df


def build_design_matrix(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series, List[str], ColumnTransformer, List[int]]:
    """Assemble feature matrix, targets, and preprocessing objects."""
    numeric_base = ["floor_area_sqm", "Latitude", "Longitude", "t_idx"]
    distance_cols = sorted(col for col in df.columns if col.startswith("dist_m_"))
    numeric_features = [col for col in numeric_base + distance_cols if col in df.columns]

    cat_features = ["town"] if "town" in df.columns else []

    feature_columns = numeric_features + cat_features
    if not numeric_features:
        raise ValueError("No numeric features detected for modeling.")

    X = df[feature_columns].copy()
    if "town" in X.columns:
        X["town"] = X["town"].astype(str).fillna("Unknown")

    y = df["resale_price"].astype(float)
    y_log = np.log1p(y)

    ohe = make_one_hot_encoder()
    numeric_pipeline = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", ohe, cat_features),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

    cat_indices = [
        X.columns.get_loc(col) for col in cat_features
    ]  # used for CatBoost

    return X, y, y_log, feature_columns, preprocess, cat_indices


def time_split(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """Create non-leaky temporal split with 80/20 within each quarter."""
    if "qtr" not in df.columns or "month_dt" not in df.columns:
        raise KeyError("DataFrame must contain 'qtr' and 'month_dt' columns for splitting.")

    train_indices: List[int] = []
    test_indices: List[int] = []

    for _, part in df.groupby("qtr", sort=True):
        part = part.sort_values(["month_dt"]).copy()
        idx = part.index.to_numpy()
        if len(idx) == 0:
            continue

        ranks = (np.arange(1, len(idx) + 1) / len(idx))
        train_mask = ranks <= 0.8

        train_indices.extend(idx[train_mask])
        test_indices.extend(idx[~train_mask])

    if not train_indices or not test_indices:
        raise ValueError("Temporal split failed; verify sufficient data per quarter.")

    return np.array(sorted(train_indices)), np.array(sorted(test_indices))


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p: Optional[int],
) -> Dict[str, float]:
    """Compute regression metrics including adjusted R²."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    n = len(y_true)
    if p is not None and n > p + 1:
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    else:
        adj_r2 = np.nan

    return {"MAE": mae, "RMSE": rmse, "R2": r2, "adj_R2": adj_r2}


def get_feature_names(estimator) -> Optional[np.ndarray]:
    """Extract feature names after preprocessing if available."""
    if hasattr(estimator, "named_steps"):
        preprocess = estimator.named_steps.get("preprocess")
        if preprocess is not None and hasattr(preprocess, "get_feature_names_out"):
            return preprocess.get_feature_names_out()
    if hasattr(estimator, "feature_names_in_"):
        return estimator.feature_names_in_
    return None


def evaluate_model(
    model,
    X_train: pd.DataFrame,
    y_train_target: pd.Series,
    X_test: pd.DataFrame,
    y_test_target: pd.Series,
    y_train_raw: pd.Series,
    y_test_raw: pd.Series,
    is_log_target: bool,
    fit_params: Optional[dict] = None,
    max_splits: int = 3,
) -> Tuple[dict, dict, any]:
    """Run time-series CV and holdout evaluation for a given model."""
    fit_params = fit_params or {}

    X_train_sorted = X_train.sort_values("t_idx")
    y_train_target_sorted = y_train_target.loc[X_train_sorted.index]
    y_train_raw_sorted = y_train_raw.loc[X_train_sorted.index]

    cv_results: List[dict] = []
    n_train = len(X_train_sorted)
    splits = min(max_splits, max(1, n_train - 1))

    if splits >= 2:
        tscv = TimeSeriesSplit(n_splits=splits)
        for train_idx, val_idx in tscv.split(X_train_sorted):
            est = clone(model)
            est.fit(
                X_train_sorted.iloc[train_idx],
                y_train_target_sorted.iloc[train_idx],
                **fit_params,
            )
            preds = est.predict(X_train_sorted.iloc[val_idx])
            if is_log_target:
                preds = np.expm1(preds)
            y_true = y_train_raw_sorted.iloc[val_idx]

            feature_names = get_feature_names(est)
            p = len(feature_names) if feature_names is not None else None
            metrics = compute_metrics(y_true.to_numpy(), preds, p)
            cv_results.append(metrics)
    else:
        cv_results.append({"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "adj_R2": np.nan})

    cv_metrics = {key: float(np.nanmean([fold[key] for fold in cv_results])) for key in cv_results[0]}

    final_estimator = clone(model)
    final_estimator.fit(X_train_sorted, y_train_target_sorted, **fit_params)

    preds_test = final_estimator.predict(X_test)
    if is_log_target:
        preds_test = np.expm1(preds_test)

    feature_names = get_feature_names(final_estimator)
    p = len(feature_names) if feature_names is not None else None
    test_metrics = compute_metrics(y_test_raw.to_numpy(), preds_test, p)

    metrics_bundle = {
        "cv": {
            "MAE": cv_metrics["MAE"],
            "RMSE": cv_metrics["RMSE"],
            "R2": cv_metrics["R2"],
            "adj_R2": cv_metrics["adj_R2"],
        },
        "test": test_metrics,
    }

    return metrics_bundle, {"preds_test": preds_test, "feature_names": feature_names}, final_estimator


def get_linear_pipeline(preprocess: ColumnTransformer) -> Pipeline:
    """Create linear regression pipeline."""
    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    )


def get_rf_pipeline(preprocess: ColumnTransformer) -> Pipeline:
    """Create random forest pipeline."""
    rf = RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", rf),
        ]
    )


def tune_rf_model(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
) -> Pipeline:
    """Run a lightweight randomized search for Random Forest hyperparameters."""
    if len(X) < 200:
        return pipeline

    param_distributions = {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": [None, "sqrt", 0.8],
    }

    n_splits = min(3, max(2, len(X) - 1))
    n_splits = min(n_splits, len(X) - 1)
    if n_splits < 2:
        return pipeline

    tscv = TimeSeriesSplit(n_splits=n_splits)

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions,
        n_iter=10,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    search.fit(X, y)
    return search.best_estimator_


def tune_xgb_model(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
) -> Pipeline:
    """Run randomized search for XGBoost model."""
    if len(X) < 200:
        return pipeline

    param_distributions = {
        "model__n_estimators": [300, 600, 900],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.03, 0.1, 0.2],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
        "model__reg_alpha": [0.0, 0.01, 0.1],
        "model__reg_lambda": [1.0, 2.0, 5.0],
    }

    n_splits = min(3, max(2, len(X) - 1))
    n_splits = min(n_splits, len(X) - 1)
    if n_splits < 2:
        return pipeline

    tscv = TimeSeriesSplit(n_splits=n_splits)

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions,
        n_iter=12,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    search.fit(X, y)
    return search.best_estimator_


def tune_cat_model(
    model: CatBoostRegressor,
    X: pd.DataFrame,
    y: pd.Series,
    cat_features: List[int],
) -> CatBoostRegressor:
    """Randomized hyperparameter search for CatBoost."""
    if len(X) < 200:
        base_model = clone(model)
        base_model.fit(X, y, cat_features=cat_features, verbose=0)
        return base_model

    param_grid = {
        "depth": [6, 8, 10],
        "learning_rate": [0.03, 0.06, 0.1],
        "l2_leaf_reg": [3, 5, 7, 9],
        "iterations": [400, 800, 1200],
    }

    n_splits = min(3, max(2, len(X) - 1))
    n_splits = min(n_splits, len(X) - 1)
    if n_splits < 2:
        base_model = clone(model)
        base_model.fit(X, y, cat_features=cat_features, verbose=0)
        return base_model

    tscv = TimeSeriesSplit(n_splits=n_splits)

    best_score = np.inf
    base_params = model.get_params()
    best_params = base_params.copy()
    rng = np.random.default_rng(RANDOM_STATE)
    keys = list(param_grid.keys())

    for _ in range(12):
        params = {key: rng.choice(param_grid[key]) for key in keys}
        candidate_params = base_params.copy()
        candidate_params.update(params)
        temp_model = CatBoostRegressor(**candidate_params)

        fold_losses: List[float] = []
        for train_idx, val_idx in tscv.split(X):
            temp_model.fit(
                X.iloc[train_idx],
                y.iloc[train_idx],
                cat_features=cat_features,
                verbose=0,
            )
            preds = temp_model.predict(X.iloc[val_idx])
            rmse = math.sqrt(mean_squared_error(y.iloc[val_idx], preds))
            fold_losses.append(rmse)

        avg_rmse = float(np.mean(fold_losses))
        if avg_rmse < best_score:
            best_score = avg_rmse
            best_params = candidate_params.copy()

    tuned_model = CatBoostRegressor(**best_params)
    tuned_model.fit(X, y, cat_features=cat_features, verbose=0)
    return tuned_model


def summarize_linear_coefficients(
    model: Pipeline,
    top_n: int = 10,
) -> pd.DataFrame:
    """Return sorted coefficient summary for linear model."""
    preprocess = model.named_steps["preprocess"]
    reg = model.named_steps["model"]

    feature_names = preprocess.get_feature_names_out()
    coefs = reg.coef_

    coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    coef_df["abs_coef"] = coef_df["coefficient"].abs()
    coef_df = coef_df.sort_values("coefficient", ascending=False)

    top_positive = coef_df.head(top_n)
    top_negative = coef_df.tail(top_n)

    summary = pd.concat([top_positive, top_negative]).drop(columns="abs_coef")
    return summary.reset_index(drop=True)


def extract_feature_importance(
    estimator,
    feature_names: Iterable[str],
) -> pd.Series:
    """Extract feature importances from tree-based models."""
    if hasattr(estimator, "named_steps"):
        model = estimator.named_steps["model"]
    else:
        model = estimator

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        return pd.Series(importances, index=feature_names).sort_values(ascending=False)

    if hasattr(model, "get_feature_importance"):
        importances = model.get_feature_importance()
        return pd.Series(importances, index=feature_names).sort_values(ascending=False)

    raise AttributeError("Model does not expose feature importance.")


def report_outputs(
    results_df: pd.DataFrame,
    best_models: Dict[str, dict],
    lin_coef_df: Optional[pd.DataFrame],
    feat_importances: Dict[str, pd.Series],
) -> None:
    results_ordered = results_df.sort_values("test_RMSE").reset_index(drop=True)
    print(results_ordered)

    if lin_coef_df is not None and not lin_coef_df.empty:
        print("\nLinear model top coefficients:")
        print(lin_coef_df)

    for model_name in ("RandomForest", "XGBoost", "CatBoost"):
        importance = feat_importances.get(model_name)
        if importance is None:
            continue
        print(f"\n{model_name} top features:")
        print(importance.head(20))


def main(arg_list: Optional[List[str]] = None):
    args = parse_args(arg_list)
    artifacts_dir = Path(args.artifacts_dir).expanduser()
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    artifact_paths = get_artifact_paths(artifacts_dir)

    cached = None if args.refresh else load_artifacts(artifact_paths)
    if cached is not None:
        results_df = cached["results_df"]
        best_models = cached["best_models"]
        lin_coef_df = cached["lin_coef_df"]
        feat_importances = cached["feat_importances"]
        splits = cached["splits"]
        X_train = splits["X_train"]
        X_test = splits["X_test"]
        y_train = splits["y_train"]
        y_test = splits["y_test"]

        report_outputs(results_df, best_models, lin_coef_df, feat_importances)

        globals().update(
            {
                "results_df": results_df,
                "best_models": best_models,
                "lin_coef_df": lin_coef_df,
                "feat_importances": feat_importances,
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test,
            }
        )
        print(f"\nLoaded cached artifacts from {artifacts_dir}")
        return

    df = load_and_prepare_data(DATA_PATH)
    X, y, y_log, feature_cols, preprocess_template, cat_indices = build_design_matrix(df)

    train_idx, test_idx = time_split(df)

    X_train = X.loc[train_idx].copy()
    X_test = X.loc[test_idx].copy()
    y_train = y.loc[train_idx].copy()
    y_test = y.loc[test_idx].copy()
    y_train_log = y_log.loc[train_idx].copy()
    y_test_log = y_log.loc[test_idx].copy()

    X_train = X_train.sort_values("t_idx")
    y_train = y_train.loc[X_train.index]
    y_test = y_test.sort_index()
    y_train_log = y_train_log.loc[X_train.index]
    y_test_log = y_test_log.sort_index()

    # Ensure test aligns with X_test sorting by t_idx
    X_test = X_test.sort_values("t_idx")
    y_test = y_test.loc[X_test.index]
    y_test_log = y_test_log.loc[X_test.index]

    results: List[dict] = []
    best_models: Dict[str, dict] = {}
    feat_importances: Dict[str, pd.Series] = {}
    lin_coef_df: Optional[pd.DataFrame] = None

    model_builders = {
        "Linear": get_linear_pipeline,
        "RandomForest": get_rf_pipeline,
    }

    tuned_pipelines: Dict[Tuple[str, str], any] = {}

    for target_scale in ("raw", "log"):
        is_log = target_scale == "log"
        y_train_target = y_train_log if is_log else y_train
        y_test_target = y_test_log if is_log else y_test

        for model_name, builder in model_builders.items():
            preprocess = clone(preprocess_template)
            pipeline = builder(preprocess)

            X_for_search = X_train.copy()
            y_for_search = y_train_target.copy()

            if model_name == "RandomForest":
                pipeline = tune_rf_model(pipeline, X_for_search, y_for_search)

            metrics_bundle, extras, fitted_model = evaluate_model(
                pipeline,
                X_train,
                y_train_target,
                X_test,
                y_test_target,
                y_train,
                y_test,
                is_log,
            )

            tuned_pipelines[(model_name, target_scale)] = fitted_model

            results.append(
                {
                    "model": model_name,
                    "target_scale": target_scale,
                    "cv_MAE": metrics_bundle["cv"]["MAE"],
                    "cv_RMSE": metrics_bundle["cv"]["RMSE"],
                    "cv_R2": metrics_bundle["cv"]["R2"],
                    "test_MAE": metrics_bundle["test"]["MAE"],
                    "test_RMSE": metrics_bundle["test"]["RMSE"],
                    "test_R2": metrics_bundle["test"]["R2"],
                    "test_adj_R2": metrics_bundle["test"]["adj_R2"],
                }
            )

    if XGB_AVAILABLE:
        preprocess = clone(preprocess_template)
        xgb_pipeline = Pipeline(
            steps=[
                ("preprocess", preprocess),
                (
                    "model",
                    XGBRegressor(
                        objective="reg:squarederror",
                        tree_method="hist",
                        random_state=RANDOM_STATE,
                        n_estimators=400,
                        learning_rate=0.1,
                        max_depth=6,
                        subsample=0.8,
                        colsample_bytree=0.8,
                    ),
                ),
            ]
        )
        for target_scale in ("raw", "log"):
            is_log = target_scale == "log"
            y_train_target = y_train_log if is_log else y_train
            y_test_target = y_test_log if is_log else y_test

            tuned_xgb = tune_xgb_model(clone(xgb_pipeline), X_train, y_train_target)
            metrics_bundle, extras, fitted_model = evaluate_model(
                tuned_xgb,
                X_train,
                y_train_target,
                X_test,
                y_test_target,
                y_train,
                y_test,
                is_log,
            )

            tuned_pipelines[("XGBoost", target_scale)] = fitted_model
            results.append(
                {
                    "model": "XGBoost",
                    "target_scale": target_scale,
                    "cv_MAE": metrics_bundle["cv"]["MAE"],
                    "cv_RMSE": metrics_bundle["cv"]["RMSE"],
                    "cv_R2": metrics_bundle["cv"]["R2"],
                    "test_MAE": metrics_bundle["test"]["MAE"],
                    "test_RMSE": metrics_bundle["test"]["RMSE"],
                    "test_R2": metrics_bundle["test"]["R2"],
                    "test_adj_R2": metrics_bundle["test"]["adj_R2"],
                }
            )
    else:
        print("⚠️  xgboost not available; skipping XGBoost model.")

    if CAT_AVAILABLE and cat_indices:
        base_cat = CatBoostRegressor(
            depth=8,
            learning_rate=0.05,
            iterations=800,
            loss_function="RMSE",
            random_state=RANDOM_STATE,
            verbose=0,
        )

        for target_scale in ("raw", "log"):
            is_log = target_scale == "log"
            y_train_target = y_train_log if is_log else y_train
            y_test_target = y_test_log if is_log else y_test

            tuned_cat = tune_cat_model(clone(base_cat), X_train, y_train_target, cat_indices)

            metrics_bundle, extras, fitted_model = evaluate_model(
                tuned_cat,
                X_train,
                y_train_target,
                X_test,
                y_test_target,
                y_train,
                y_test,
                is_log,
                fit_params={"cat_features": cat_indices},
            )

            tuned_pipelines[("CatBoost", target_scale)] = fitted_model
            results.append(
                {
                    "model": "CatBoost",
                    "target_scale": target_scale,
                    "cv_MAE": metrics_bundle["cv"]["MAE"],
                    "cv_RMSE": metrics_bundle["cv"]["RMSE"],
                    "cv_R2": metrics_bundle["cv"]["R2"],
                    "test_MAE": metrics_bundle["test"]["MAE"],
                    "test_RMSE": metrics_bundle["test"]["RMSE"],
                    "test_R2": metrics_bundle["test"]["R2"],
                    "test_adj_R2": metrics_bundle["test"]["adj_R2"],
                }
            )
    else:
        if not CAT_AVAILABLE:
            print("⚠️  catboost not available; skipping CatBoost model.")
        else:
            print("⚠️  No categorical indices supplied; skipping CatBoost model.")

    results_df = pd.DataFrame(results).sort_values("test_RMSE").reset_index(drop=True)

    for model_name in results_df["model"].unique():
        subset = results_df[results_df["model"] == model_name]
        best_row = subset.loc[subset["test_RMSE"].idxmin()]
        key = (model_name, best_row["target_scale"])
        best_models[model_name] = {
            "estimator": tuned_pipelines[key],
            "target_scale": best_row["target_scale"],
        }

    if ("Linear", "raw") in tuned_pipelines or ("Linear", "log") in tuned_pipelines:
        chosen = best_models.get("Linear")
        if chosen:
            lin_model = chosen["estimator"]
            lin_coef_df = summarize_linear_coefficients(lin_model)

    for model_name in ("RandomForest", "XGBoost", "CatBoost"):
        chosen = best_models.get(model_name)
        if not chosen:
            continue
        estimator = chosen["estimator"]
        feature_names = get_feature_names(estimator)
        if feature_names is None:
            feature_names = X_train.columns
        importance = extract_feature_importance(estimator, feature_names)
        feat_importances[model_name] = importance

    report_outputs(results_df, best_models, lin_coef_df, feat_importances)

    save_artifacts(
        artifact_paths,
        results_df,
        best_models,
        lin_coef_df,
        feat_importances,
        X_train,
        X_test,
        y_train,
        y_test,
    )
    print(f"\nSaved artifacts to {artifacts_dir}")

    globals().update(
        {
            "results_df": results_df,
            "best_models": best_models,
            "lin_coef_df": lin_coef_df,
            "feat_importances": feat_importances,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
        }
    )


if __name__ == "__main__":
    main()


          model target_scale        cv_MAE        cv_RMSE     cv_R2  \
0       XGBoost          log  38446.970864   57627.695474  0.906742   
1       XGBoost          raw  38297.371569   56837.441483  0.909460   
2      CatBoost          raw  38532.355529   57001.546433  0.908769   
3      CatBoost          log  38674.023219   58014.437866  0.905293   
4  RandomForest          raw  39337.169414   58977.214018  0.902799   
5  RandomForest          log  39742.502009   59717.903643  0.900327   
6        Linear          raw  84038.935246  116823.379581  0.625885   
7        Linear          log  82948.816638  126738.562847  0.556301   

       test_MAE      test_RMSE   test_R2  test_adj_R2  
0  30527.649637   44480.025494  0.946208     0.946135  
1  30652.476129   44581.352685  0.945962     0.945889  
2  30983.862931   44802.263688  0.945426          NaN  
3  30800.378952   44888.447883  0.945215          NaN  
4  32161.447196   48440.128828  0.936203     0.936117  
5  32306.060994   48693.

NameError: name 'results_df' is not defined